# Analyze contrastive training results
This notebook is used to analyze the results of fine-tuning CLIP and of extending Grounding DINO.

### 0. Import libraries

In [ ]:
import copy
import json

import polars as pl
import plotly.express as px
import plotly.graph_objects as go

RESULTS_PATH = "../../experiments/contrastive_training/"
COLORS = ["#cd968e", "#acb0e0", "#aecbdc", "#bcd5c3", "#bfbfbf", "#000000"]

### 1. Prepare data and display loss evolution

#### 1.1. CLIP

In [ ]:
with open(f"{RESULTS_PATH}clip_full_1e_7.json", "r") as f:
    loss_evolution_clip = json.load(f)

train_val_loss_clip = copy.deepcopy(loss_evolution_clip["losses"])
del train_val_loss_clip["test_loss"]

In [ ]:
train_val_loss_clip_df = (
    pl.from_dict(train_val_loss_clip)
    .rename({"train_losses": "Train", "val_losses": "Val"})
    .with_columns(pl.Series(list(range(1, len(train_val_loss_clip["train_losses"]) + 1))).alias("Epoch"))
)
train_val_loss_clip_df

In [ ]:
fig = px.line(
    train_val_loss_clip_df,
    x="Epoch",
    y=["Train", "Val"],
    labels={"value": "Contrastive Loss Value", "variable": "Set"},
    title="Evolution of the Contrastive Loss",
    markers=True,
    color_discrete_sequence=COLORS,
)

fig.update_traces(
    selector=dict(name="Train"),
    marker=dict(symbol="circle", size=12)
)
fig.update_traces(
    selector=dict(name="Val"),
    marker=dict(symbol="square", size=12)
)

fig.add_trace(
    go.Line(
        x=[train_val_loss_clip_df["Epoch"].max()],
        y=[loss_evolution_clip["losses"]["test_loss"]],
        name="Test",
        marker=dict(size=14, color="black", symbol="x"),
    )
)

fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=[1] + list(range(0, train_val_loss_clip_df["Epoch"].max() + 1, 5)),
        range=[0.1, 20.8],
    )
)


font_size = 18
fig.update_layout(
    font=dict(size=font_size),
    title_font_size=font_size,
    xaxis=dict(
        title_font_size=font_size,
        tickfont_size=font_size
    ),
    yaxis=dict(
        title_font_size=font_size,
        tickfont_size=font_size
    ),
    legend=dict(
        font_size=font_size
    ),
    width=1600,  # Set the desired width in pixels
    height=500,  # Set the desired height in pixels
)

fig.show()
fig.write_image("fine_tuning_performance_plot.png", scale=6)


#### 1.2. Added encoder

In [ ]:
with open(f"{RESULTS_PATH}mean_pooling_frozen_encoder_dif_lr.json", "r") as f:
    loss_evolution_enc = json.load(f)

train_val_loss_enc = copy.deepcopy(loss_evolution_enc["losses"])
del train_val_loss_enc["test_loss"]

In [ ]:
train_val_loss_enc_df = (
    pl.from_dict(train_val_loss_enc)
    .rename({"train_losses": "Train", "val_losses": "Val"})
    .with_columns(pl.Series(list(range(1, len(train_val_loss_enc["train_losses"]) + 1))).alias("Epoch"))
)
train_val_loss_enc_df

In [ ]:
fig = px.line(
    train_val_loss_enc_df,
    x="Epoch",
    y=["Train", "Val"],
    labels={"value": "Contrastive Loss Value", "variable": "Set"},
    title="",
    markers=True,
    color_discrete_sequence=COLORS,
)


fig.update_traces(
    selector=dict(name="Train"),
    marker=dict(symbol="circle", size=12)
)
fig.update_traces(
    selector=dict(name="Val"),
    marker=dict(symbol="square", size=12)
)

fig.add_trace(
    go.Line(
        x=[train_val_loss_enc_df["Epoch"].max()],
        y=[loss_evolution_enc["losses"]["test_loss"]],
        name="Test",
        marker=dict(size=14, color="black", symbol="x"),
    )
)

fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=[1] + list(range(0, train_val_loss_enc_df["Epoch"].max() + 1, 5)),
        range=[0.1, 30.8],
    )
)

font_size = 18
fig.update_layout(
    font=dict(size=font_size),
    title_font_size=font_size,
    xaxis=dict(
        title_font_size=font_size,
        tickfont_size=font_size
    ),
    yaxis=dict(
        title_font_size=font_size,
        tickfont_size=font_size
    ),
    legend=dict(
        font_size=font_size
    ),
    width=1600,  # Set the desired width in pixels
    height=500,  # Set the desired height in pixels
)

fig.show()
fig.write_image("fine_tuning_performance_plot.png", scale=6)


#### 1.3. Projecting embeddings

In [ ]:
with open(f"{RESULTS_PATH}projections_text_embedding_backbone.json", "r") as f:
    loss_evolution_backbone = json.load(f)

train_val_backbone = copy.deepcopy(loss_evolution_backbone["losses"])
del train_val_backbone["test_loss"]

In [ ]:
train_val_loss_backbone_df = (
    pl.from_dict(train_val_backbone)
    .rename({"train_losses": "Train", "val_losses": "Val"})
    .with_columns(pl.Series(list(range(1, len(train_val_backbone["train_losses"]) + 1))).alias("Epoch"))
)
train_val_loss_backbone_df

In [ ]:
with open(f"{RESULTS_PATH}projections_text_embedding_enhanced.json", "r") as f:
    loss_evolution_enhanced = json.load(f)

train_val_enhanced = copy.deepcopy(loss_evolution_enhanced["losses"])
del train_val_enhanced["test_loss"]

In [ ]:
train_val_loss_enhanced_df = (
    pl.from_dict(train_val_enhanced)
    .rename({"train_losses": "Train", "val_losses": "Val"})
    .with_columns(pl.Series(list(range(1, len(train_val_enhanced["train_losses"]) + 1))).alias("Epoch"))
)
train_val_loss_enhanced_df

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
    x=train_val_loss_backbone_df["Epoch"].to_numpy(),
    y=train_val_loss_backbone_df["Train"].to_numpy(),
    name="Train Backbone",
    mode='lines+markers',
    line=dict(color=COLORS[0], width=2),
    marker=dict(symbol="circle", size=12), 
))

fig.add_trace(
    go.Scatter(
    x=train_val_loss_backbone_df["Epoch"].to_numpy(),
    y=train_val_loss_backbone_df["Val"].to_numpy(),
    name="Val Backbone",
    mode='lines+markers',
    line=dict(color=COLORS[1], width=2),
    marker=dict(symbol="square", size=12), 
))

fig.add_trace(
    go.Line(
        x=[train_val_loss_backbone_df["Epoch"].max()],
        y=[loss_evolution_backbone["losses"]["test_loss"]],
        name="Test Backbone",
        marker=dict(size=14, color="black", symbol="x"),
    )
)

fig.add_trace(
    go.Scatter(
    x=train_val_loss_enhanced_df["Epoch"].to_numpy(),
    y=train_val_loss_enhanced_df["Train"].to_numpy(),
    name="Train Enhanced",
    mode='lines+markers',
    line=dict(color=COLORS[2], width=2),
    marker=dict(symbol="diamond", size=12), 
))

fig.add_trace(
    go.Scatter(
    x=train_val_loss_enhanced_df["Epoch"].to_numpy(),
    y=train_val_loss_enhanced_df["Val"].to_numpy(),
    name="Val Enhanced",
    mode='lines+markers',
    line=dict(color="gray", width=2),
    marker=dict(symbol="triangle-left", size=14), 
))

fig.add_trace(
    go.Line(
        x=[train_val_loss_enhanced_df["Epoch"].max()],
        y=[loss_evolution_enhanced["losses"]["test_loss"]],
        name="Test Enhanced",
        marker=dict(size=14, color="black", symbol="cross"),
    )
) 

fig.update_layout(
    title="Evolution of Contrastive Loss",
    xaxis_title="Epoch",
    yaxis_title="Contrastive Loss Value",
    legend_title="Set & Embeddings Type", 
    xaxis=dict(
        tickmode="array",
        tickvals=[1] + list(range(0, train_val_loss_backbone_df["Epoch"].max() + 1, 5)),
        range=[0.1, 30.8],
    )
)

font_size = 18
fig.update_layout(
    font=dict(size=font_size),
    title_font_size=font_size,
    xaxis=dict(
        title_font_size=font_size,
        tickfont_size=font_size
    ),
    yaxis=dict(
        title_font_size=font_size,
        tickfont_size=font_size
    ),
    legend=dict(
        font_size=font_size
    ),
    width=1600,  # Set the desired width in pixels
    height=500,  # Set the desired height in pixels
)

fig.show()
fig.write_image("fine_tuning_performance_plot.png", scale=6)
